# SVI: fitting a well-behaved volatility smile

This notebook builds up the SVI (Stochastic Volatility Inspired) smile
parametrization from motivation through theory to a validated fitter, working
entirely on synthetic data where we control the ground truth. The final
section, fitting real CBOE quotes, is added separately once the machinery here
is trusted.

The order is deliberate: understand the object and its failure modes on data
we generate ourselves, so that when real market noise arrives we can tell a
data problem from a code problem.

## 1. The problem: from noisy quotes to a usable surface

The market gives us option quotes at a scatter of discrete strikes for one
expiry. Convert each to an implied volatility, then to *total implied variance*
$w = \sigma_{BS}^2\,\tau$, and plot against *log-moneyness* $k = \log(K/F)$.
The result is a point cloud, the "smile," and it is noisy, especially in the
illiquid wings.

We want a smooth curve threaded through that cloud. Three reasons, in
increasing order of importance:

1. **Denoising.** Downstream model calibration should fit clean signal, not
   bid-ask bounce and stale strikes.
2. **No-arbitrage.** Raw quotes, particularly in the wings, can be mutually
   inconsistent, implying negative probability densities. Calibrating a model
   straight to those asks it to fit prices no arbitrage-free model can produce.
   SVI gives a target that is guaranteed arbitrage-free.
3. **Continuity.** Quotes exist only at discrete strikes. A continuous curve
   lets us evaluate the market's implied variance at any $k$, which model
   calibration needs.

So the goal is a smooth, arbitrage-free, continuously-evaluable representation
of the smile. One caution to carry throughout: SVI can only produce shapes SVI
can produce, so if the fit systematically misses the same region on every
slice, that is real structure being discarded, not noise removed. Watch the
residual pattern, not just its size.

In [1]:
# Setup. Import the certified module and plotting.
import numpy as np
import matplotlib.pyplot as plt

from calibration.svi import (
    SVIParams,
    svi_raw,
    svi_raw_first,
    svi_raw_second,
    durrleman_g,
    fit_slice,
    verify_fit,
)

# A representative equity-smile slice, used throughout as our ground truth.
# Negative rho and m reflect real equity skew (left wing steeper, vertex just
# below the money).
TRUE = SVIParams(a=0.04, b=0.4, rho=-0.3, m=-0.05, sigma=0.15)

In [3]:
# Illustrate reason 1: a jagged "market" smile vs the clean SVI curve.
# Generate a clean SVI slice, add noise to mimic quote noise, plot both.
#
# Build:
#   k_quotes = coarse grid, say 15 points over [-0.4, 0.4]
#   w_true   = svi_raw(k_quotes, TRUE)
#   w_noisy  = w_true + small gaussian noise (seed it, sigma ~ 1e-3)
#   k_dense  = fine grid for the smooth curve
# Plot: w_noisy as scatter, svi_raw(k_dense, TRUE) as a line.
# Point to make in the plot: the line is what we want, the dots are what we get.

## 2. The parametrization

Raw SVI, per maturity slice, in total variance against log-moneyness:

$$w(k) = a + b\Big(\rho(k-m) + \sqrt{(k-m)^2 + \sigma^2}\Big)$$

Five parameters, each with a clean geometric role:

- $a$ shifts the whole slice vertically (the overall variance level).
- $b$ sets the angle between the two asymptotes (overall wing steepness).
- $\rho$ tilts the smile (the skew); negative $\rho$ makes the left wing steeper,
  which is the equity signature.
- $m$ translates the smile horizontally (where the vertex sits).
- $\sigma$ controls how rounded the vertex is; $\sigma \to 0$ gives a sharp kink
  at $k = m$.

As $k \to \pm\infty$ the curve is asymptotically linear, with slopes
$b(1+\rho)$ on the right and $b(1-\rho)$ on the left. That linearity is not
cosmetic: it is exactly what Lee's moment formula demands, that implied
variance grow at most linearly in $|k|$. SVI is built to respect that, which is
why extrapolating it into the wings does not blow up.

The name is worth taking literally. SVI is *inspired by* stochastic volatility
models (Gatheral showed the large-time Heston smile has this form) but there is
no SDE underneath. It is a static parametrization, a clean interpolant of one
day's quotes. It carries no dynamics, which is precisely why it serves as the
target that dynamic models like Heston and SABR calibrate against.

In [2]:
# Show each parameter moving, so a reader SEES its effect.
# Five small subplots (or a 2x3 grid). In each, plot the TRUE slice as a
# reference line, then overlay 2-3 variants with one parameter perturbed.
#
# e.g. the 'b' panel: TRUE, plus TRUE with b=0.2 and b=0.6, others fixed.
# Do the same for a, rho, m, sigma.
# k_dense over [-0.6, 0.6] so the wings show.
# The rho and sigma panels are the most instructive: rho tilts, sigma rounds.

## 3. Why total variance, not implied vol

The choice of $w = \sigma_{BS}^2\tau$ over plain implied vol is not arbitrary.
The no-arbitrage conditions are natural in total variance.

Calendar arbitrage (roughly, that a longer-dated option cannot be worth less
than a shorter-dated one at the same moneyness) reduces in total variance to a
plain monotonicity condition: $w(k)$ must be non-decreasing in maturity
$\tau$. Total variance can only accumulate as you add time. In implied-vol
space that same condition is a messier expression and easy to implement wrong.

Log-moneyness against the *forward* (not spot) is the companion choice: it
centres each slice on the at-the-money-forward point and makes slices at
different maturities directly comparable.

## 4. Arbitrage within a slice, and the function that detects it

A curve can hug the quotes beautifully and still be illegal. Butterfly
arbitrage corresponds to a negative implied probability density, and a fit can
produce one in the wings while looking perfect near the money.

The density implied by a slice, written in log-moneyness, factors into a
strictly positive part times a single function Gatheral calls $g$:

$$g(k) = \left(1 - \frac{k\,w'}{2w}\right)^2
        - \frac{w'^2}{4}\left(\frac{1}{w} + \frac14\right)
        + \frac{w''}{2}$$

Because everything multiplying $g$ in the density is positive, **butterfly
arbitrage is exactly $g(k) < 0$**. One scalar function, a sign check.

Reading the three terms tells us where violations live. The first is a square,
always non-negative. For raw SVI the third is $w''/2 > 0$ everywhere. So only
the middle term can pull $g$ negative, and it does so where $w$ is small and
$w'$ is steep. That means short-dated slices on the steep put wing, not the far
wing. Two consequences we will use: fit the shortest maturities first (failures
surface before a surface is built on top of them), and check $g$ most densely
near the money.

The asymptotic version of the same condition recovers Lee's bound. As
$k \to \infty$, $g(\infty) = \tfrac14 - \tfrac{\beta^2}{16}$ with
$\beta = b(1+\rho)$, so $g \ge 0$ forces $b(1+\rho) \le 2$. The density
condition and the wing-slope bound are the same statement.

In [ ]:
# Make the arbitrage check visual: a clean slice vs a broken one.
#
# Left: TRUE slice. Plot w(k) on top, g(k) below, shade g < 0 regions.
#       g should be positive everywhere -> no shading.
# Right: a slice that violates Lee, e.g. SVIParams(a=0.04, b=5.0, rho=-0.7,
#        m=-0.05, sigma=0.15). Confirm b*(1+|rho|) > 2. Plot w and g.
#        g should dip below zero in a wing -> shaded region.
#
# durrleman_g is already imported. Two-row subplot per slice, or a 2x2 grid.
# The teaching point: the checker fires on the illegal slice and not the legal
# one. A checker that never fires is not a checker.

## 5. Fitting: the two-stage reduction

Fitting five parameters directly is hard because they are strongly correlated.
Near the money the data reveals only a level, a slope, and a curvature, three
observable combinations, while we are trying to pin down five parameters. In
particular $\{a, b, \sigma\}$ collapse onto about two of those combinations, and
a naive optimizer from a random start lands in local minima.

The Zeliade reduction unties the knot. Substitute $y = (k-m)/\sigma$ and rename
$c = b\sigma$, $d = \rho b\sigma$. Then

$$w = a + d\,y + c\sqrt{y^2+1}$$

which for *fixed* $(m,\sigma)$ is **linear** in $(a, d, c)$. So we split the
problem into two nested loops:

- **Inner**: given $(m,\sigma)$, solve for $(a,d,c)$ by constrained linear least
  squares. Fast, reliable, no local minima. The constraints ($b \ge 0$,
  $|\rho| \le 1$, $w \ge 0$, the Lee bound) are all linear in the reduced
  coordinates.
- **Outer**: search over just $(m,\sigma)$, two dimensions, with Nelder-Mead
  from several spread-out starts.

Five correlated dimensions become two clean geometric ones with a cheap exact
solve inside. The arbitrage penalty rides on the outer objective as a squared
hinge on $g$, so the inner problem stays linear while the search is steered
away from arbitrageable curves.

In [5]:
# Illustrate the identifiability point that motivates the reduction.
# Show two DIFFERENT parameter sets that produce nearly the SAME curve near the
# money, to make "badly identified" concrete.
#
# Take TRUE, and a perturbed set where a is lowered and b*sigma raised to
# roughly compensate (recall level ~ a + b*sigma). Plot both over a modest k
# range: near the money they nearly coincide, they separate only in the wings.
# This is why the fit needs wing data and why the reduction helps.

## 6. Validation: ground truth first

Before trusting the fitter on real data, we confirm it recovers known
parameters from data it generated itself. If it cannot fit a noiseless smile it
produced, nothing downstream is trustworthy.

The test below synthesizes $w$ from `TRUE`, fits, and checks that the fitted
*curve* matches. We assert on the curve rather than exact parameter recovery
because the identifiability just discussed means the recovered parameters can
sit slightly off the true ones while the curve is near-perfect. The curve is
what matters downstream.

In [6]:
# The ground-truth demonstration (mirrors test_fit_slice_recovers_known_params).
#
#   k = np.linspace(-0.4, 0.4, 25)
#   w = svi_raw(k, TRUE)
#   result = fit_slice(k, w, lam=0.0)
#
# Then:
#   - print result.params vs TRUE, side by side
#   - print result.rmse, result.arbitrage_free
#   - plot the noiseless data and the fitted curve overlaid (should be
#     indistinguishable)
#   - run verify_fit(result.params, (k.min(), k.max())) and print the three
#     returns: clean on data, clean on extrapolation, min_g
#
# Teaching point: the fitter is certified here. Any misbehaviour on the real
# CBOE slice tomorrow is therefore a data or plumbing problem, not a fitter bug.
# The variable is isolated.

## 7. Real CBOE data

*To be added next session.* With the fitter certified above, the remaining work
is the bridge from raw quotes to a clean $(k, w)$ array: pull a CBOE chain,
apply the OTM convention, use the parity-implied forward as canonical spot,
convert quotes to total variance, and build weights from bid-ask spreads. Then
fit real slices, gate each with `verify_fit`, and fit multiple maturities to
check for calendar arbitrage across the surface.